## 01 — Config & Dimension Generation (customers, merchants)
Run this first. It creates customers and merchants CSVs and lands them
in the UC Volume landing path. Downstream notebooks read these to enforce
referential integrity in the fact table.


In [ ]:
from datetime import datetime as _dt

dbutils.widgets.text("catalog", "northstar_dev")
dbutils.widgets.text("schema", "raw")
dbutils.widgets.text("volume", "landing")
dbutils.widgets.text("num_customers", "50000")
dbutils.widgets.text("num_merchants", "9000")
dbutils.widgets.text("random_seed", "42")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
NUM_CUSTOMERS = int(dbutils.widgets.get("num_customers"))
NUM_MERCHANTS = int(dbutils.widgets.get("num_merchants"))
SEED = int(dbutils.widgets.get("random_seed"))

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

_y, _m = _dt.now().strftime("%Y"), _dt.now().strftime("%m")
CUSTOMERS_PATH = f"{VOLUME_ROOT}/customers/{_y}/{_m}"
MERCHANTS_PATH = f"{VOLUME_ROOT}/merchants/{_y}/{_m}"

print(f"Catalog.Schema.Volume = {CATALOG}.{SCHEMA}.{VOLUME}")
print(f"Volume root           = {VOLUME_ROOT}")


## Pre-flight: create catalog/schema/volume if they don't already exist
Safe to run even if these already exist (IF NOT EXISTS).


In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# dbutils.fs.mkdirs(CUSTOMERS_PATH)
# dbutils.fs.mkdirs(MERCHANTS_PATH)
# dbutils.fs.mkdirs(f"{VOLUME_ROOT}/transactions")
# dbutils.fs.mkdirs(f"{VOLUME_ROOT}/device_session_logs")


## Customers
Faker runs on the driver for PII-shaped fields, distributed via a Pandas UDF
so it scales past a few thousand rows without single-threaded bottlenecks.
`risk_band` and `kyc_status` are intentionally mutable downstream (SCD2 target).


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, TimestampType
)
import uuid
from datetime import datetime, timedelta

SEGMENT_WEIGHTS = {"retail": 0.70, "premium": 0.22, "business": 0.08}
RISK_BAND_WEIGHTS = {"low": 0.65, "medium": 0.25, "high": 0.10}
KYC_WEIGHTS = {"verified": 0.90, "pending": 0.07, "rejected": 0.03}

def generate_customers_partition(pdf_iter):
    """
    Pandas UDF generator pattern: each partition gets its own Faker instance
    seeded deterministically off the partition's first row index, so the whole
    job is reproducible across re-runs even though it's distributed.
    """
    for pdf in pdf_iter:
        seed_offset = int(pdf["partition_seed"].iloc[0])
        fake = Faker()
        Faker.seed(SEED + seed_offset)
        rng = np.random.default_rng(SEED + seed_offset)

        n = len(pdf)
        full_names, dobs, signup_dates, segments = [], [], [], []
        risk_bands, addresses, kyc_statuses, last_updated = [], [], [], []

        for _ in range(n):
            full_names.append(fake.name())
            dobs.append(fake.date_of_birth(minimum_age=18, maximum_age=85))
            signup_dt = fake.date_between(start_date="-3y", end_date="-1d")
            signup_dates.append(signup_dt)
            segments.append(rng.choice(list(SEGMENT_WEIGHTS.keys()),
                                        p=list(SEGMENT_WEIGHTS.values())))
            risk_bands.append(rng.choice(list(RISK_BAND_WEIGHTS.keys()),
                                          p=list(RISK_BAND_WEIGHTS.values())))
            addresses.append(fake.address().replace("\n", ", "))
            kyc_statuses.append(rng.choice(list(KYC_WEIGHTS.keys()),
                                            p=list(KYC_WEIGHTS.values())))
            # last_updated_ts sits any time between signup and now -> supports SCD2 testing
            days_since_signup = (datetime.now().date() - signup_dt).days
            last_updated.append(
                datetime.combine(signup_dt, datetime.min.time())
                + timedelta(days=int(rng.integers(0, max(days_since_signup, 1))))
            )

        out = pdf.copy()
        out["full_name"] = full_names
        out["dob"] = dobs
        out["signup_date"] = signup_dates
        out["segment"] = segments
        out["risk_band"] = risk_bands
        out["address"] = addresses
        out["kyc_status"] = kyc_statuses
        out["last_updated_ts"] = last_updated
        yield out.drop(columns=["partition_seed"])


# Seed frame: customer_id + a partition_seed column so each Spark partition's
# Faker instance is independently and reproducibly seeded.
NUM_PARTITIONS = 32
seed_pdf = pd.DataFrame({
    "customer_id": [f"CUST-{i:07d}" for i in range(1, NUM_CUSTOMERS + 1)],
})
seed_pdf["partition_seed"] = (seed_pdf.index % NUM_PARTITIONS)

customers_seed_df = spark.createDataFrame(seed_pdf).repartition(NUM_PARTITIONS, "partition_seed")

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("full_name", StringType(), False),
    StructField("dob", DateType(), False),
    StructField("signup_date", DateType(), False),
    StructField("segment", StringType(), False),
    StructField("risk_band", StringType(), False),
    StructField("address", StringType(), False),
    StructField("kyc_status", StringType(), False),
    StructField("last_updated_ts", TimestampType(), False),
])

customers_df = customers_seed_df.mapInPandas(generate_customers_partition, schema=customers_schema)
# customers_df.cache()
print(f"Generated {customers_df.count():,} customers")
display(customers_df.limit(5))


## Merchants
Smaller table, generated in a single Pandas pass (driver) since 8-10k rows
doesn't need distribution. MCC codes drawn from a realistic fixed pool so
they correlate sensibly with merchant category.


In [ ]:
MCC_CATEGORY_MAP = {
    "5411": "Grocery Stores",
    "5812": "Restaurants",
    "5912": "Pharmacies",
    "5732": "Electronics Stores",
    "4111": "Transportation - Transit",
    "5541": "Service Stations",
    "5651": "Family Clothing Stores",
    "7011": "Hotels/Motels",
    "5999": "Specialty Retail",
    "4899": "Cable/Streaming Services",
    "6011": "ATM / Financial",
    "5311": "Department Stores",
    "5812": "Restaurants",
    "7995": "Gambling",  # intentionally higher risk category
    "4814": "Telecom",
}
MCC_CODES = list(MCC_CATEGORY_MAP.keys())

COUNTRY_POOL = ["US", "GB", "CA", "DE", "FR", "AU", "SG", "NL", "IE", "MX"]
COUNTRY_WEIGHTS = [0.45, 0.12, 0.10, 0.08, 0.06, 0.05, 0.04, 0.04, 0.03, 0.03]

fake = Faker()
Faker.seed(SEED)
rng = np.random.default_rng(SEED)

merchant_rows = []
for i in range(1, NUM_MERCHANTS + 1):
    mcc = rng.choice(MCC_CODES)
    country = rng.choice(COUNTRY_POOL, p=COUNTRY_WEIGHTS)
    onboarding = fake.date_between(start_date="-5y", end_date="-1d")
    # Gambling / high-MCC merchants skew toward higher risk scores -> gives
    # downstream fraud/risk marts a believable signal to surface later.
    base_risk = rng.normal(loc=35, scale=15)
    if mcc == "7995":
        base_risk += 30
    risk_score = float(np.clip(base_risk, 0, 100))

    merchant_rows.append({
        "merchant_id": f"MERC-{i:06d}",
        "merchant_name": fake.company(),
        "mcc_code": mcc,
        "category": MCC_CATEGORY_MAP[mcc],
        "country": country,
        "onboarding_date": onboarding,
        "merchant_risk_score": round(risk_score, 2),
    })

merchants_pdf = pd.DataFrame(merchant_rows)
merchants_df = spark.createDataFrame(merchants_pdf)
# merchants_df.cache()
print(f"Generated {merchants_df.count():,} merchants")
display(merchants_df.limit(5))


## Land as CSV into the UC Volume
Single-file-per-table CSVs (coalesce) since these are dimension tables —
small enough that multi-part output would just add friction for Autoloader.


In [ ]:
(customers_df.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(CUSTOMERS_PATH)
)

(merchants_df.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(MERCHANTS_PATH)
)

print(f"Customers landed at: {CUSTOMERS_PATH}")
print(f"Merchants landed at: {MERCHANTS_PATH}")

## Hand off customer_id / merchant_id pools to downstream notebooks
Saved as small Delta lookup tables (not the volume CSVs) purely so the
transaction generator notebook can read just the ID + a few skew-relevant
columns without re-parsing full CSVs.


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.datagen_scratch")

(customers_df.select("customer_id", "signup_date")
    .write.mode("overwrite")
    .saveAsTable(f"{CATALOG}.datagen_scratch.customer_ids"))

(merchants_df.select("merchant_id", "mcc_code", "country", "merchant_risk_score")
    .write.mode("overwrite")
    .saveAsTable(f"{CATALOG}.datagen_scratch.merchant_ids"))

print("Scratch lookup tables written for downstream generation.")